# Day 14 — SQL Case Study: Business Analysis

This notebook reproduces an end-to-end retail business analysis entirely in **SQLite** through
Python, then integrates the results with **Pandas** and **Seaborn**.

**Workflow**

1. Create and populate a five-table schema (Customers, Products, Orders, OrderItems, Payments).
2. Run the analytical queries (KPIs, profit, top customers, segments, inactive customers,
   geography, monthly trend, payment reconciliation).
3. Bring an SQL result into Pandas and visualize it.
4. Close the connection.

> All data below is **synthetic sample data** created for this exercise.


## Step 1 — Create and populate the database

**Objective:** Build the five-table schema and load the sample data.

In [ ]:
import sqlite3
import pandas as pd

conn = sqlite3.connect(":memory:")
cur = conn.cursor()

# --- Schema ---
cur.executescript("""
CREATE TABLE Customers (
    CustomerID   INTEGER PRIMARY KEY,
    CustomerName TEXT, City TEXT, State TEXT, Segment TEXT, SignupDate TEXT
);
CREATE TABLE Products (
    ProductID   TEXT PRIMARY KEY,
    ProductName TEXT, Category TEXT, Price INTEGER, Cost INTEGER
);
CREATE TABLE Orders (
    OrderID    TEXT PRIMARY KEY,
    CustomerID INTEGER, OrderDate TEXT, Status TEXT
);
CREATE TABLE OrderItems (
    OrderItemID INTEGER PRIMARY KEY,
    OrderID TEXT, ProductID TEXT, Quantity INTEGER, UnitPrice INTEGER
);
CREATE TABLE Payments (
    PaymentID   TEXT PRIMARY KEY,
    OrderID TEXT, PaymentMode TEXT, PaidAmount INTEGER, PaymentDate TEXT
);
""")

print("Schema created.")

In [ ]:
# --- Data ---
customers = [
    (1, "Asha Reddy",   "Hyderabad", "Telangana",   "Premium", "2024-02-10"),
    (2, "Ravi Kumar",   "Mumbai",    "Maharashtra", "Regular", "2024-05-15"),
    (3, "Imran Khan",   "Pune",      "Maharashtra", "Premium", "2023-11-20"),
    (4, "Divya Rao",    "Delhi",     "Delhi",       "Regular", "2025-01-05"),
    (5, "Karan Mehta",  "Hyderabad", "Telangana",   "Premium", "2024-08-30"),
    (6, "Sneha Iyer",   "Bengaluru", "Karnataka",   "Regular", "2024-03-12"),
    (7, "Vikram Singh", "Delhi",     "Delhi",       "Premium", "2025-03-01"),
    (8, "Meena Nair",   "Bengaluru", "Karnataka",   "Regular", "2024-09-25"),
]

products = [
    ("P1", "Laptop Pro",          "Electronics", 80000, 62000),
    ("P2", "Smartphone X",        "Electronics", 45000, 34000),
    ("P3", "Wireless Mouse",      "Accessories",  1500,   800),
    ("P4", "Mechanical Keyboard", "Accessories",  4000,  2500),
    ("P5", "Noise Headphones",    "Audio",       12000,  8000),
    ("P6", "Smartwatch",          "Wearables",   20000, 14000),
]

orders = [
    ("O-1001", 1, "2026-01-12", "Completed"), ("O-1002", 2, "2026-01-20", "Completed"),
    ("O-1003", 3, "2026-02-05", "Completed"), ("O-1004", 1, "2026-02-18", "Completed"),
    ("O-1005", 4, "2026-03-02", "Completed"), ("O-1006", 5, "2026-03-15", "Completed"),
    ("O-1007", 3, "2026-04-10", "Completed"), ("O-1008", 6, "2026-04-22", "Cancelled"),
    ("O-1009", 1, "2026-05-08", "Completed"), ("O-1010", 7, "2026-05-19", "Completed"),
    ("O-1011", 2, "2026-06-03", "Completed"), ("O-1012", 5, "2026-06-14", "Completed"),
]

order_items = [
    (1,"O-1001","P1",1,80000),(2,"O-1001","P3",1,1500),(3,"O-1002","P2",1,45000),
    (4,"O-1003","P1",1,80000),(5,"O-1003","P5",1,12000),(6,"O-1004","P6",1,20000),
    (7,"O-1005","P2",1,45000),(8,"O-1005","P4",1,4000),(9,"O-1006","P1",1,80000),
    (10,"O-1007","P5",2,12000),(11,"O-1008","P3",1,1500),(12,"O-1009","P2",1,45000),
    (13,"O-1009","P3",2,1500),(14,"O-1010","P1",1,80000),(15,"O-1010","P6",1,20000),
    (16,"O-1011","P5",1,12000),(17,"O-1011","P4",1,4000),(18,"O-1012","P6",2,20000),
]

payments = [
    ("PAY-1", "O-1001", "Card",       81500, "2026-01-12"),
    ("PAY-2", "O-1002", "UPI",        45000, "2026-01-20"),
    ("PAY-3", "O-1003", "Card",       92000, "2026-02-05"),
    ("PAY-4", "O-1004", "UPI",        20000, "2026-02-18"),
    ("PAY-5", "O-1005", "Card",       49000, "2026-03-02"),
    ("PAY-6", "O-1006", "NetBanking", 80000, "2026-03-15"),
    ("PAY-7", "O-1007", "UPI",        24000, "2026-04-10"),
    ("PAY-8", "O-1009", "Card",       48000, "2026-05-08"),
    ("PAY-9", "O-1010", "UPI",        95000, "2026-05-19"),
    ("PAY-10","O-1011", "Card",       16000, "2026-06-03"),
]

cur.executemany("INSERT INTO Customers  VALUES (?,?,?,?,?,?)", customers)
cur.executemany("INSERT INTO Products   VALUES (?,?,?,?,?)",   products)
cur.executemany("INSERT INTO Orders     VALUES (?,?,?,?)",     orders)
cur.executemany("INSERT INTO OrderItems VALUES (?,?,?,?,?)",   order_items)
cur.executemany("INSERT INTO Payments   VALUES (?,?,?,?,?)",   payments)
conn.commit()

print("Database created and populated.")

## Step 2 — Run the analysis queries

**Objective:** Execute each analytical query and review the result.

A small helper `run()` returns every query result as a Pandas DataFrame.

In [ ]:
def run(sql):
    """Execute SQL against the in-memory DB and return a DataFrame."""
    return pd.read_sql(sql, conn)

### Baseline KPIs (Phase 2)
Completed orders, active customers, and total revenue.

In [ ]:
run("""
SELECT COUNT(DISTINCT o.OrderID)        AS CompletedOrders,
       COUNT(DISTINCT o.CustomerID)     AS ActiveCustomers,
       SUM(oi.Quantity * oi.UnitPrice)  AS TotalRevenue
FROM Orders o
JOIN OrderItems oi ON o.OrderID = oi.OrderID
WHERE o.Status = 'Completed'
""")

### Profit by category (Phase 5)
Revenue and profit grouped by product category.

In [ ]:
run("""
SELECT p.Category,
       SUM(oi.Quantity * oi.UnitPrice)               AS Revenue,
       SUM(oi.Quantity * (oi.UnitPrice - p.Cost))    AS Profit
FROM Orders o
JOIN OrderItems oi ON o.OrderID   = oi.OrderID
JOIN Products   p  ON oi.ProductID = p.ProductID
WHERE o.Status = 'Completed'
GROUP BY p.Category
ORDER BY Profit DESC
""")

### Top customers by revenue
The five highest-spending customers among completed orders.

In [ ]:
run("""
SELECT c.CustomerName,
       c.City,
       SUM(oi.Quantity * oi.UnitPrice) AS Revenue
FROM Orders o
JOIN OrderItems oi ON o.OrderID    = oi.OrderID
JOIN Customers  c  ON o.CustomerID = c.CustomerID
WHERE o.Status = 'Completed'
GROUP BY c.CustomerID
ORDER BY Revenue DESC
LIMIT 5
""")

### Revenue by customer segment
Compare the Premium and Regular segments.

In [ ]:
run("""
SELECT c.Segment,
       COUNT(DISTINCT o.OrderID)       AS Orders,
       SUM(oi.Quantity * oi.UnitPrice) AS Revenue
FROM Orders o
JOIN OrderItems oi ON o.OrderID    = oi.OrderID
JOIN Customers  c  ON o.CustomerID = c.CustomerID
WHERE o.Status = 'Completed'
GROUP BY c.Segment
ORDER BY Revenue DESC
""")

### Inactive customers
Customers who have never placed a completed order.

In [ ]:
run("""
SELECT c.CustomerID, c.CustomerName, c.City, c.Segment
FROM Customers c
WHERE c.CustomerID NOT IN (
    SELECT DISTINCT CustomerID
    FROM Orders
    WHERE Status = 'Completed'
)
ORDER BY c.CustomerID
""")

### Revenue by geography
Revenue grouped by state and city.

In [ ]:
run("""
SELECT c.State,
       c.City,
       SUM(oi.Quantity * oi.UnitPrice) AS Revenue
FROM Orders o
JOIN OrderItems oi ON o.OrderID    = oi.OrderID
JOIN Customers  c  ON o.CustomerID = c.CustomerID
WHERE o.Status = 'Completed'
GROUP BY c.State, c.City
ORDER BY Revenue DESC
""")

### Monthly revenue trend
Revenue per month across the analysis window.

In [ ]:
run("""
SELECT strftime('%Y-%m', o.OrderDate) AS Month,
       SUM(oi.Quantity * oi.UnitPrice) AS Revenue
FROM Orders o
JOIN OrderItems oi ON o.OrderID = oi.OrderID
WHERE o.Status = 'Completed'
GROUP BY Month
ORDER BY Month
""")

### Payment reconciliation
Compare each completed order's computed value against the recorded payment.
A non-zero `Difference` (or a missing payment) flags an order that needs follow-up.

In [ ]:
run("""
SELECT o.OrderID,
       SUM(oi.Quantity * oi.UnitPrice)              AS OrderValue,
       p.PaidAmount,
       p.PaidAmount - SUM(oi.Quantity * oi.UnitPrice) AS Difference
FROM Orders o
JOIN OrderItems oi ON o.OrderID = oi.OrderID
LEFT JOIN Payments p ON o.OrderID = p.OrderID
WHERE o.Status = 'Completed'
GROUP BY o.OrderID
ORDER BY o.OrderID
""")

## Step 3 — Combine SQL with Pandas and visualization

**Objective:** Bring an SQL result into Pandas and visualize it — SQL aggregates in the
database, Pandas receives the summary, and Seaborn renders the chart for the report.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

cat = run("""
SELECT p.Category,
       SUM(oi.Quantity * oi.UnitPrice) AS Revenue
FROM Orders o
JOIN OrderItems oi ON o.OrderID    = oi.OrderID
JOIN Products   p  ON oi.ProductID = p.ProductID
WHERE o.Status = 'Completed'
GROUP BY p.Category
ORDER BY Revenue DESC
""")

plt.figure(figsize=(7, 4))
sns.barplot(data=cat, x="Category", y="Revenue")
plt.title("Revenue by Category")
plt.ylabel("Revenue")
plt.tight_layout()
plt.show()

### Bonus: monthly revenue trend as a line chart

In [ ]:
trend = run("""
SELECT strftime('%Y-%m', o.OrderDate) AS Month,
       SUM(oi.Quantity * oi.UnitPrice) AS Revenue
FROM Orders o
JOIN OrderItems oi ON o.OrderID = oi.OrderID
WHERE o.Status = 'Completed'
GROUP BY Month
ORDER BY Month
""")

plt.figure(figsize=(7, 4))
sns.lineplot(data=trend, x="Month", y="Revenue", marker="o")
plt.title("Monthly Revenue Trend")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## Step 4 — Close the connection

In [ ]:
conn.close()
print("Analysis complete; connection closed.")